# Maze Solver - Solution Validation Analysis

This notebook validates whether LLM-generated solutions are actually correct,
not just exact matches to the training solutions.

**Key Questions:**
1. How many predictions match exactly?
2. How many predictions are valid alternative solutions?
3. What types of errors does the model make?
4. Is the model truly overfitting or just being creative?

In [87]:
import sys
import json
import torch
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

# Add src to path
sys.path.append('./src')

if 'solution_validator' in sys.modules:
    del sys.modules['solution_validator']

# Import our modules
from model import ResNetGPT2PrefixModel
from tokenizer import SimpleTokenizer
from dataset import MazeDataset, collate_fn
from solution_validator import validate_solution, evaluate_with_validation, print_evaluation_results
from data_utils import load_maze_dataset, print_dataset_info

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 1. Load Trained Model

In [88]:
# Load checkpoint
checkpoint = torch.load('models/resnet_gpt2_prefix.pth', map_location=device)

# Initialize tokenizer
tokenizer = SimpleTokenizer()

# Initialize model with EXACT config from training
model = ResNetGPT2PrefixModel(
    vocab_size=checkpoint['vocab_size'],
    gpt2_hidden_size=128,  # Your model used 64-dim
    num_layers=2,
    num_attention_heads=2,
    num_prefix_tokens=checkpoint['num_prefix_tokens'],
    dropout=0.4
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(f"✓ Model loaded successfully")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

C:\Users\seena\AppData\Local\Temp\ipykernel_2396\2024777241.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('models/resnet_gpt2_prefix.pth', map_

✓ Model loaded successfully
Parameters: 11,952,320


## 2. Load Test Data & Create Maze Grid Lookup

In [89]:
# Load test data
with open('data/test_sequences.json', 'r') as f:
    test_entries = json.load(f)

print(f"Loaded {len(test_entries)} test mazes")

Loaded 2 test mazes


In [91]:
def load_maze_grid(image_path, grid_rows=7, grid_cols=7):
    """
    Convert maze image to binary grid (0=path, 1=wall).
    
    Handles colored start (green) and goal (blue) squares.
    """
    # Load as RGB to detect colors
    img_rgb = Image.open(image_path).convert('RGB')
    img_array_rgb = np.array(img_rgb)
    
    # Also load grayscale for regular cells
    img_gray = Image.open(image_path).convert('L')
    img_array_gray = np.array(img_gray)
    
    height, width = img_array_gray.shape
    cell_height = height // grid_rows
    cell_width = width // grid_cols
    
    grid = np.zeros((grid_rows, grid_cols), dtype=int)
    
    for row in range(grid_rows):
        for col in range(grid_cols):
            # Sample from center of cell
            center_y = row * cell_height + cell_height // 2
            center_x = col * cell_width + cell_width // 2
            
            # Get RGB values
            r, g, b = img_array_rgb[center_y, center_x]
            
            # Check if it's green (start) or blue (goal) - treat as path
            is_green = (g > 100 and g > r and g > b)  # Green dominant
            is_blue = (b > 100 and b > r and b > g)   # Blue dominant
            
            if is_green or is_blue:
                # Colored squares are paths
                grid[row, col] = 0
            else:
                # Use grayscale for regular cells
                pixel_value = img_array_gray[center_y, center_x]
                grid[row, col] = 1 if pixel_value < 127 else 0
    
    return grid



# Load test data with metadata
test_entries, metadata = load_maze_dataset('data/test_sequences.json')

# Print dataset info
print_dataset_info(metadata, len(test_entries), dataset_name="Test")

# Extract grid size from metadata
GRID_SIZE = metadata['grid_size']
ROWS = metadata['rows']
COLS = metadata['cols']

print(f"✓ Using grid size: {ROWS}×{COLS}")
# # Reload grids with the fixed function
# GRID_SIZE = 7

maze_grids = {}
for entry in test_entries:
    maze_id = entry['id']
    img_path = entry['image']
    maze_grids[maze_id] = load_maze_grid(img_path, grid_rows=GRID_SIZE, grid_cols=GRID_SIZE)

print(f"✓ Reloaded {len(maze_grids)} maze grids with color handling")

# Verify maze 2 now
sample_grid = maze_grids[test_entries[2]['id']]
print(f"\nMaze 2 grid (fixed):")
print(sample_grid)
print(f"\nStart (0,6): {sample_grid[6, 0]} (should be 0)")
print(f"Goal (6,0): {sample_grid[0, 6]} (should be 0)")


TEST SET INFO
Grid size:       7×7
Total entries:   9250
Variations:      50
Seed:            641
Start position:  (0, 6)
Goal position:   (6, 0)

✓ Using grid size: 7×7
✓ Reloaded 9250 maze grids with color handling

Maze 2 grid (fixed):
[[0 0 0 0 0 1 0]
 [0 1 1 1 0 0 0]
 [1 1 1 1 1 0 0]
 [0 1 0 0 0 0 1]
 [1 1 1 0 0 1 1]
 [1 0 0 0 1 0 1]
 [0 0 1 0 0 0 1]]

Start (0,6): 0 (should be 0)
Goal (6,0): 0 (should be 0)


## 3. Create Test DataLoader

In [ ]:
# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create dataset
test_dataset = MazeDataset(test_entries, tokenizer, transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
)

print(f"✓ Test loader ready with {len(test_loader)} batches")

✓ Test loader ready with 290 batches


## 4. Run Validation Analysis

In [ ]:
# Run comprehensive evaluation
test_results = evaluate_with_validation(
    model=model,
    data_loader=test_loader,
    device=device,
    tokenizer=tokenizer,
    maze_grids=maze_grids
)

# Print results
print_evaluation_results(test_results, dataset_name="Test")


TEST SET RESULTS - DETAILED ANALYSIS
Total mazes evaluated: 9250

Exact Match Accuracy:   2854/9250 (30.9%)
Valid Solution Rate:    8000/9250 (86.5%)
Invalid Solutions:      1250/9250 (13.5%)

Creative Solutions:     5146/9250 (55.6%)
  ↳ Valid paths that differ from training solution

Sample Results:
----------------------------------------------------------------------

Maze 0: ✓ VALID
  Expected:  R U R R U R U R U R U U
  Predicted: R R U R U R U R U R U U

Maze 1: ✗ INVALID
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R R R U R U R U U
  Failure: hit wall at position (3, 5)

Maze 2: ✓ EXACT
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R U R U R U R U U

Maze 3: ✓ VALID
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R U R U R U R U U

Maze 4: ✓ EXACT
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R U R U R U R U U
----------------------------------------------------------------------


## 5. Detailed Error Analysis

In [ ]:
# Categorize results
exact_matches = [d for d in test_results['details'] if d['exact_match']]
creative_solutions = [d for d in test_results['details'] 
                      if d['valid_solution'] and not d['exact_match']]
invalid_solutions = [d for d in test_results['details'] if not d['valid_solution']]

print(f"\n📊 BREAKDOWN:")
print(f"  Exact matches:       {len(exact_matches)}")
print(f"  Creative solutions:  {len(creative_solutions)}")
print(f"  Invalid solutions:   {len(invalid_solutions)}")

# Analyze failure modes
if invalid_solutions:
    hit_wall = sum(1 for d in invalid_solutions 
                   if d['validation'] and d['validation'].get('hit_wall', False))
    out_of_bounds = sum(1 for d in invalid_solutions 
                        if d['validation'] and d['validation'].get('out_of_bounds', False))
    invalid_token = sum(1 for d in invalid_solutions 
                        if d['validation'] and d['validation'].get('invalid_token', False))
    didnt_reach = len(invalid_solutions) - hit_wall - out_of_bounds - invalid_token
    
    print(f"\n❌ FAILURE MODES:")
    print(f"  Hit wall:            {hit_wall}")
    print(f"  Out of bounds:       {out_of_bounds}")
    print(f"  Invalid token:       {invalid_token}")
    print(f"  Didn't reach goal:   {didnt_reach}")


📊 BREAKDOWN:
  Exact matches:       2854
  Creative solutions:  5146
  Invalid solutions:   1250

❌ FAILURE MODES:
  Hit wall:            1204
  Out of bounds:       46
  Invalid token:       0
  Didn't reach goal:   0


## 6. Visualize Examples

In [ ]:
# Show creative solutions (valid but different)
print("\n🎨 CREATIVE SOLUTIONS (Valid but different from training):")
print("=" * 70)
for i, detail in enumerate(creative_solutions[:10]):
    print(f"\nMaze {detail['maze_id']}:")
    print(f"  Expected:  {' '.join(detail['expected'])}")
    print(f"  Predicted: {' '.join(detail['predicted'])}")
    print(f"  Length: Expected={len(detail['expected'])}, Predicted={len(detail['predicted'])}")


🎨 CREATIVE SOLUTIONS (Valid but different from training):

Maze 0:
  Expected:  R U R R U R U R U R U U
  Predicted: R R U R U R U R U R U U
  Length: Expected=12, Predicted=12

Maze 3:
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R U R U R U R U U
  Length: Expected=12, Predicted=12

Maze 7:
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R U R U R U R U U
  Length: Expected=12, Predicted=12

Maze 8:
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R U R U R U R U U
  Length: Expected=12, Predicted=12

Maze 9:
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R R U U U R R U U
  Length: Expected=12, Predicted=12

Maze 10:
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R R U U U R R U U
  Length: Expected=12, Predicted=12

Maze 12:
  Expected:  R U R R U R U R U R U U
  Predicted: R R U R U R U R U U R U
  Length: Expected=12, Predicted=12

Maze 14:
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R R U U R U R U U
  Length: Expected=12,

In [ ]:
# Show failed solutions
print("\n❌ INVALID SOLUTIONS (First 10):")
print("=" * 70)
for i, detail in enumerate(invalid_solutions[:10]):
    val = detail['validation']
    if val:
        reason = "hit wall" if val.get('hit_wall') else \
                 ("out of bounds" if val.get('out_of_bounds') else \
                 ("invalid token" if val.get('invalid_token') else "didn't reach goal"))
        
        print(f"\nMaze {detail['maze_id']}: FAILED - {reason}")
        print(f"  Expected:  {' '.join(detail['expected'])}")
        print(f"  Predicted: {' '.join(detail['predicted'])}")
        print(f"  Final position: {val['final_position']}")
        print(f"  Moves made: {val['num_moves']}")


❌ INVALID SOLUTIONS (First 10):

Maze 1: FAILED - hit wall
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R R R U R U R U U
  Final position: (3, 5)
  Moves made: 4

Maze 5: FAILED - hit wall
  Expected:  R U R R U R U R U R U U
  Predicted: U R R U U U U R R R R U
  Final position: (4, 1)
  Moves made: 9

Maze 11: FAILED - hit wall
  Expected:  R U R R U R U R U R U U
  Predicted: R U R U R U R U U R R U
  Final position: (3, 4)
  Moves made: 5

Maze 13: FAILED - hit wall
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R U U R U R U U U
  Final position: (4, 3)
  Moves made: 7

Maze 18: FAILED - hit wall
  Expected:  R U R R U R U R U R U U
  Predicted: U R U U R R R U U R R U
  Final position: (1, 5)
  Moves made: 2

Maze 34: FAILED - out of bounds
  Expected:  R U R R U R U R U R U U
  Predicted: R U R R U R U R R R U U
  Final position: (6, 3)
  Moves made: 9

Maze 35: FAILED - hit wall
  Expected:  R U R R U R U R U R U U
  Predicted: U R R R U U R U R R U U
  Fi

## 7. Summary & Next Steps

In [ ]:
print("\n" + "=" * 70)
print("SUMMARY & INTERPRETATION")
print("=" * 70)

exact_pct = test_results['exact_match_pct']
valid_pct = test_results['valid_solution_pct']
creative_pct = 100 * len(creative_solutions) / test_results['total']

print(f"\nOriginal metric (exact match):  {exact_pct:.1f}%")
print(f"True performance (valid solns): {valid_pct:.1f}%")
print(f"Creative solutions:             {creative_pct:.1f}%")

improvement = valid_pct - exact_pct
print(f"\n📈 Performance underestimated by: {improvement:.1f}%")

if valid_pct >= 60:
    print("\n✅ EXCELLENT: Model is generalizing well!")
    print("   → Model learned spatial reasoning, not just memorization")
    print("   → High creative solution rate shows true understanding")
    print("   → Consider: Keep current architecture, maybe train longer")
elif valid_pct >= 45:
    print("\n✓ GOOD: Model shows decent generalization")
    print("   → Model has learned some spatial reasoning")
    print("   → Consider: Add more regularization or data augmentation")
else:
    print("\n⚠️  NEEDS WORK: Model struggles with generalization")
    print("   → Model may still be overfitting to training patterns")
    print("   → Consider: Stronger regularization or smaller model")

print("=" * 70)


SUMMARY & INTERPRETATION

Original metric (exact match):  30.9%
True performance (valid solns): 86.5%
Creative solutions:             55.6%

📈 Performance underestimated by: 55.6%

✅ EXCELLENT: Model is generalizing well!
   → Model learned spatial reasoning, not just memorization
   → High creative solution rate shows true understanding
   → Consider: Keep current architecture, maybe train longer
